#### implicitly_wait (隱式等待)

#### 練習：設定隱式等待 (Implicit Wait) 以處理動態載入元素

#### 教學目標：理解 implicitly_wait() 的輪詢機制，並與 time.sleep() 的強制等待做區分

In [2]:
import os
import time
from selenium import webdriver
from selenium.webdriver.common.by import By

# 取得 HTML 檔案的絕對路徑 (檔案在同一目錄下)
file_path = "file://" + os.path.abspath("implicit_wait_demo.html")

# 初始化 Driver
driver = webdriver.Chrome()

try:
    # TODO: 設定全域的「隱式等待」時間為 10 秒
    # 提示：呼叫 driver 的 implicitly_wait() 方法，並傳入整數 10
    # 原理：若找不到元素，Driver 不會立刻拋出錯誤，而是每隔一段時間輪詢 DOM，直到元素出現或超過 10 秒超時。
    # (已知目標元素需等待 8 秒才會出現，因此 10 秒的寬限期能確保成功抓取)
    driver.implicitly_wait(5)

    print(f"[{time.strftime('%H:%M:%S')}] 開始載入頁面...")
    driver.get(file_path)

    print(f"[{time.strftime('%H:%M:%S')}] 嘗試尋找元素 (id='dynamic_element')...")
    
    # TODO: 尋找網頁中動態生成的元素
    # 提示：請使用 find_element() 方法，並透過 By.ID 策略尋找特徵值為 "dynamic_element" 的元素
    element = driver.find_element(By.ID, 'dynamic_element')
    
    # 驗證結果與提取資料
    print(f"[{time.strftime('%H:%M:%S')}] 成功找到元素！")
    
    # TODO: 取得元素的文字內容，並印出
    # 提示：呼叫 element 物件的 text 屬性
    print(f"元素文字內容: { ... }")
    
    # TODO: 驗證文字內容是否符合預期
    # 提示：使用 assert 關鍵字比對剛剛提取出的 element.text 是否等於 expected_text
    expected_text = "美好的事物值得等待，它會在該來的時候來"
    assert element.text == expected_text
    
    # TODO: 對該元素執行點擊操作 (將觸發頁面上的 alert 視窗)
    element.click()
    
    time.sleep(10) # 為了讓大家肉眼看清楚 alert 視窗的結果，此處保留強制暫停

except Exception as e:
    print(f"發生錯誤: {e}")

finally:
    driver.quit()

[11:22:05] 開始載入頁面...
[11:22:05] 嘗試尋找元素 (id='dynamic_element')...
發生錯誤: Message: no such element: Unable to locate element: {"method":"css selector","selector":"[id="dynamic_element"]"}
  (Session info: chrome=147.0.7727.138); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#nosuchelementexception
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff78eaca8e5+14e45]
	chromedriver!GetHandleVerifier [0x7ff78eaca950+14eb0]
	chromedriver!(No symbol) [0x7ff78e83d6ed]
	chromedriver!(No symbol) [0x7ff78e896cfe]
	chromedriver!(No symbol) [0x7ff78e89700c]
	chromedriver!(No symbol) [0x7ff78e8e7cb7]
	chromedriver!(No symbol) [0x7ff78e8e483b]
	chromedriver!(No symbol) [0x7ff78e8890e8]
	chromedriver!(No symbol) [0x7ff78e889fc3]
	chromedriver!GetHandleVerifier [0x7ff78ede0149+32a6a9]
	chromedriver!GetHandleVerifier [0x7ff78edda715+324c75]
	chromedriver!GetHandleVerifier [0x7ff78edfc012+346572]
	chromedriver!GetHandleVerifier [0x7ff78e

我們把
implicitly_wait(10) 改為 implicitly_wait(5)，觀察 $T_{wait} < T_{load}$ 時發生的 NoSuchElementException 錯誤

顯式等待 (Explicit Wait)

In [3]:
import os
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# 1. 設定 HTML 檔案路徑
file_path = "file://" + os.path.abspath("implicit_wait_demo.html")

driver = webdriver.Chrome()
try:
    print(f"[{time.strftime('%H:%M:%S')}] 開始載入頁面...")
    driver.get(file_path)

    print(f"[{time.strftime('%H:%M:%S')}] 開始顯式等待 (Waiting for element to be clickable)...")
    
    # 2. 顯示等待 (Explicit Wait)
    # 說明：明確告知程式要等待特定的元素滿足特定的條件。
    
    # TODO: [請在此處寫下建立 WebDriverWait 物件的程式碼]
    # 預期輸入：WebDriver 實例 (driver) 與最長等待秒數 (請設定為 15 秒)。
    # 為什麼要這樣做：建立一個計時器與觀察器，避免因網頁渲染過慢導致找不到元素而拋出例外。
    wait_object = WebDriverWait(driver, 15)
    
    # TODO: [請在此處寫下預期條件與定位器 (Locator)]
    # 預期輸入：呼叫 until 方法，並傳入 EC (expected_conditions) 中的「元素可被點擊」條件。
    # 條件內需要傳入一個 Locator Tuple -> (定位策略, "目標字串")，請使用 ID 定位，目標 ID 為 "dynamic_element"。
    # 為什麼要這樣做：明確定義「等待終止的條件」，確保元素不僅存在於 DOM 樹中，且在畫面上是可見且可互動的。
    element = wait_object.until(
        EC.element_to_be_clickable((By.ID, "dynamic_element"))
    )

    print(f"[{time.strftime('%H:%M:%S')}] 條件滿足！元素已出現且可點擊。")
    
    # 3. 執行動作
    # TODO: [請在此處寫下對該元素執行的動作]
    # 預期輸入：觸發滑鼠點擊的方法。
    # 為什麼要這樣做：當確認元素狀態允許互動後，我們必須實際觸發它以測試網頁的後續行為 (例如跳出 Alert)。
    element.click()
    
    print("已點擊按鈕，觸發 Alert。")
    
    # 暫停一下看效果
    time.sleep(3) 

except Exception as e:
    print(f"[{time.strftime('%H:%M:%S')}] 發生錯誤或等待超時: {e}")

finally:
    driver.quit()

[11:25:40] 開始載入頁面...
[11:25:40] 開始顯式等待 (Waiting for element to be clickable)...
[11:25:48] 條件滿足！元素已出現且可點擊。
已點擊按鈕，觸發 Alert。
